<a href="https://colab.research.google.com/github/AlaaSoudy/NTI_ML_Tasks/blob/main/Task_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Heart Disease Prediction using K-Nearest Neighbors (KNN)

***Step_1 : Import Required Libraries***

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix , accuracy_score , classification_report
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder

***Step 2: Load the Dataset***

Dataset: [Heart Disease Prediction Dataset](https://www.kaggle.com/datasets/shyamnadhs/heart-disease-prediction-dataset)


In [ ]:
df=pd.read_csv('/content/drive/MyDrive/disease_prediction.csv')
df.head()

***Step 3: Explore the Dataset (EDA)***

In [ ]:
df.info()

Insight: No missing values were detected, and the data types are valid; only categorical features will require encoding before modeling.

In [ ]:
df.describe()

In [ ]:
df.shape

***Step_4:Data Preprocessing***

**Check Missing Values**

In [ ]:
df.isnull().sum()

Insight: No missing values were found in the dataset.

**Check Duplicate Records**

In [ ]:
df.duplicated().sum()

Insight: No duplicate records were found in the dataset.

In [ ]:
df.drop(columns='patient_id', inplace=True)

**Data Visualization**

In [ ]:
df.hist(figsize=(15,10))
plt.tight_layout()
plt.show();

In [ ]:
df.boxplot(figsize=(15,10))
plt.xticks(rotation=90)
plt.show();

In [ ]:
sns.pairplot(df, hue='disease')
plt.show();

In [ ]:
df = pd.get_dummies(df, drop_first=True)
plt.figure(figsize=(12,8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.show()

In [ ]:
df.columns

In [ ]:
# df.drop(columns=['diastolic_bp',  'gender_Male'], inplace=True)

**Handling Outliers**

In [ ]:
def remove_outliers(df, columns):
    for col in columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        df = df[(df[col] >= lower) & (df[col] <= upper)]

    return df


In [ ]:
def capping_outliers(df, columns):
    for col in columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        df[col] = np.where(df[col] < lower, lower, df[col])
        df[col] = np.where(df[col] > upper, upper, df[col])

    return df

In [ ]:
num_cols=df.select_dtypes(include='number').columns
# df=remove_outliers(df, num_cols)
capping_outliers(df, num_cols)
df.boxplot(figsize=(15,10))
plt.xticks(rotation=90)
plt.show();

In [ ]:
df.shape

**Encoding**

In [ ]:
# way_1
LE=LabelEncoder()
cat_cols=df.select_dtypes(include='object').columns
for col in cat_cols:
  df[col]=LE.fit_transform(df[col])
df.head()

In [ ]:
# Way_2
df = pd.get_dummies(df, drop_first=True)
df.head()

**Feature Scaling**

In [ ]:
# Features & Target
X = df.drop(columns=[ 'disease_Yes'])
y = df['disease_Yes']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
df=pd.DataFrame(X_train, columns=X.columns).head()
df.head()

In [ ]:
df.hist(figsize=(15,10), bins=30)
plt.tight_layout()
plt.show();

In [ ]:
print(X_train.mean(axis=0))
print(X_train.std(axis=0))

***Step_5: Build the Machine Learning Model***

In [ ]:
knn = KNeighborsClassifier(n_neighbors=9) # k=9
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
acc=accuracy_score(y_test, y_pred)
cm=confusion_matrix(y_test, y_pred)
print(f'Accuracy: {acc}')
print(cm)
print(classification_report(y_test, y_pred))


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5,4))

sns.heatmap(
    cm,
    annot=True,
    cmap="Blues",
    fmt="d"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
accuracy_list = []

k_values = range(1,21)

for k in k_values:

    model = KNeighborsClassifier(n_neighbors=k)

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    score = accuracy_score(y_test,pred)

    accuracy_list.append(score)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(k_values, accuracy_list, marker="o")

plt.title("Accuracy vs K")

plt.xlabel("K Value")

plt.ylabel("Accuracy")

plt.grid()

plt.show()